# Game monetization analysis

- **Author:** Liudmila Shcherbakova
- **Date:** 26.11.2024

### 0.1 Project goals and objectives

**Context:** The game is an online RPG where players explore the game world, complete quests, and develop their characters based on race, class, and abilities.  The game includes an in-game economy with a premium currency ("Heavenly Petals") used to purchase epic items that enhance gameplay. This currency can be earned in-game or purchased with real money, making it a key driver of revenue.

Understanding player behavior and monetization patterns is essential for improving revenue and optimizing game design.

---
**Goal:** The goal of this project is to analyze player behavior and in-game purchases to understand monetization patterns and identify factors influencing revenue generation.

Key objectives:
- Calculate the share of paying users in the game
- Analyze how payer behavior varies across player segments (e.g., race)
- Explore in-game purchase activity and transaction distribution
- Identify anomalies in purchase data (e.g., zero-value transactions)
- Evaluate popularity of in-game items
- Analyze purchasing activity and spending behavior across player groups

---
### 0.2 Data description

The project uses the **fantasy** database, which contains seven tables. The two main tables are:

- **users** – contains information about the game players, including:
  - `id` – unique identifier of a player (primary key)
  - `tech_nickname`, `class_id`, `ch_id`, `race_id`, `loc_id` – categorical fields describing the player
  - `birthdate`, `registration_dt` – date fields
  - `pers_gender` – player gender
  - `server` – game server
  - `payer` – indicator of whether the player spends real money

- **events** – records in-game purchases using the game currency "Paradise Petals", with fields:
  - `transaction_id` – unique purchase ID
  - `id` – player ID who made the purchase
  - `date`, `time` – timestamp of the purchase
  - `item_code` – code of the purchased item
  - `amount` – amount of currency spent
  - `seller_id` – identifier of the item seller

Other tables contain more detailed information about races, classes, skills, locations, and game items.

The data is mostly categorical or numerical, with some date fields. Initially, we focus on the `users` and `events` tables to understand the player base and purchase behavior, while other tables will be used later to join additional details.



---

### 0.3 Project structure

1. Exploratory data analysis
2. Ad hoc analysis
3. Final conclusions and recommendations

### 1. Exploratory data analysis

#### 1.1 Data overview
#### 1.1.1 Tables overview

First, let’s explore the structure of the `fantasy` schema and list all available tables.

```sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'fantasy';
```

**Query result**

table_name|
----------|
classes   |
country   |
events    |
items     |
race      |
skills    |
users     |

The schema contains seven tables: `classes`, `country`, `users`, `events`, `items`, `skills`, `race`. These tables store the core data about players, their attributes, and in-game activity.

#### 1.1.2  Users table exploration

The `fantasy` schema contains several tables, with `users` and `events` being the core ones for analysis.  
The `users` table stores player-level information, while `events` contains in-game purchase data using the "Paradise Petals" currency. Other tables provide additional details on players and game items.

We start the analysis with the `users` table by examining its structure, including column names, data types, and key constraints.

```sql
SELECT 
    c.table_schema,
    c.table_name,
    c.column_name,
    c.data_type,
    tc.constraint_name
FROM information_schema.columns c
LEFT JOIN information_schema.key_column_usage kcu
    ON c.table_schema = kcu.table_schema
    AND c.table_name = kcu.table_name
    AND c.column_name = kcu.column_name
LEFT JOIN information_schema.table_constraints tc
    ON kcu.constraint_name = tc.constraint_name
    AND kcu.table_schema = tc.table_schema
    AND kcu.table_name = tc.table_name
WHERE c.table_schema = 'fantasy'
  AND c.table_name = 'users';
```

**Query result**

| table_schema | table_name | column_name     | data_type         | constraint_name |
|--------------|------------|-----------------|-------------------|-----------------|
| fantasy      | users      | id              | character varying |                 |
| fantasy      | users      | tech_nickname   | character varying |                 |
| fantasy      | users      | class_id        | character varying |                 |
| fantasy      | users      | ch_id           | character varying |                 |
| fantasy      | users      | birthdate       | character varying |                 |
| fantasy      | users      | pers_gender     | character varying |                 |
| fantasy      | users      | registration_dt | character varying |                 |
| fantasy      | users      | server          | character varying |                 |
| fantasy      | users      | race_id         | character varying |                 |
| fantasy      | users      | payer           | integer           |                 |
| fantasy      | users      | loc_id          | character varying |                 |

The table consists of 11 columns, most of which store categorical (text) data.  
The `id` field serves as the primary key.  

Several fields — `class_id`, `ch_id`, `race_id`, and `loc_id` — appear to be foreign keys, indicating relationships with other tables such as `classes`, `skills`, `race`, and `country`.

This structure suggests that the `users` table can be used as a base for joining additional player attributes from related tables.

#### 1.1.3 Preview of the users table

We retrieve the first 5 rows of the users table. Additionally, we include a row_count column to capture the total number of records. This helps validate the data structure and estimate dataset size.

```sql
SELECT 
    *,
    COUNT(*) OVER () AS row_count
FROM fantasy.users
LIMIT 5;
```

**Query result**


| tech_nickname        | class_id | id         | ch_id | birthdate | pers_gender | registration_dt | server   | race_id | payer | loc_id | row_count |
|----------------------|----------|------------|-------|-----------|-------------|-----------------|----------|---------|-------|--------|-----------|
| DivineBarbarian4154  | 9RD      | 00-0037846 | JJR2  | 6/4/1994  | Male        | 1/20/2005       | server_1 | B1      | 0     | US     | 22214     |
| BoldInvoker7693      | Z3Q      | 00-0041533 | HQ9N  | 6/29/1987 | Male        | 4/8/2022        | server_1 | R2      | 0     | US     | 22214     |
| NobleAlchemist7633   | 382      | 00-0045747 | IXBW  | 7/29/1992 | Male        | 10/12/2013      | server_1 | K3      | 0     | US     | 22214     |
| SteadfastArcher8318  | ZD0      | 00-0055274 | QSUB  | 9/14/1985 | Female      | 4/10/2008       | server_1 | R2      | 0     | US     | 22214     |
| RadiantProphet353    | YC8      | 00-0076100 | HQ9N  | 4/11/1997 | Female      | 9/29/2013       | server_2 | K4      | 1     | US     | 22214     |


The dataset contains 22,214 players. At this stage, it is also useful to note potential data quality issues (e.g., date formats).

#### 1.1.4 Missing values in the users table

Since id is a primary key, duplicate rows are not expected. 

Next, we check for missing values in key fields used for analysis:
`class_id`, `ch_id`, `pers_gender`, `server`, `race_id`, `payer`, `loc_id`.

```sql
SELECT COUNT(*) AS rows_with_nulls
FROM fantasy.users
WHERE class_id IS NULL
   OR ch_id IS NULL
   OR pers_gender IS NULL
   OR server IS NULL
   OR race_id IS NULL
   OR payer IS NULL
   OR loc_id IS NULL;
```

#### Query result

| rows_with_nulls |
|-----------------|
| 0               |

There are no missing values in these fields.

#### 1.1.5 Exploring categorical features in users

The users table includes several categorical attributes: character characteristics (race, class, skills), gender, server
location. 

We analyze the distribution of players across servers by extracting unique values of server and counting records per group.

```sql
SELECT 
    server,
    COUNT(*) AS user_count
FROM fantasy.users
GROUP BY server
ORDER BY user_count DESC;
```

**Query result**


server  |user_count
--------|----------
server_1|     16715
server_2|      5499


There are two servers, with approximately 3x more players on server_1 than on server_2.

#### 1.1.6 Exploring the events table

The second key table is events, which stores in-game purchase data.

We extract: column names, data types, key constraints.

```sql
SELECT 
    c.table_schema,
    c.table_name,
    c.column_name,
    c.data_type,
    tc.constraint_name
FROM information_schema.columns c
LEFT JOIN information_schema.key_column_usage kcu
    ON c.table_schema = kcu.table_schema
    AND c.table_name = kcu.table_name
    AND c.column_name = kcu.column_name
LEFT JOIN information_schema.table_constraints tc
    ON kcu.constraint_name = tc.constraint_name
    AND kcu.table_schema = tc.table_schema
    AND kcu.table_name = tc.table_name
WHERE c.table_schema = 'fantasy'
  AND c.table_name = 'events';
```

**Query result**

| table_schema | table_name | column_name    | data_type         | constraint_name |
|--------------|------------|----------------|-------------------|-----------------|
| fantasy      | events     | transaction_id | character varying |                 |
| fantasy      | events     | id             | character varying |                 |
| fantasy      | events     | date           | character varying |                 |
| fantasy      | events     | time           | character varying |                 |
| fantasy      | events     | item_code      | integer           |                 |
| fantasy      | events     | amount         | real              |                 |
| fantasy      | events     | seller_id      | character varying |                 |

The events table contains 7 columns, mostly text-based.

- `transaction_id` - primary key
- `id` and `item_code` - foreign keys, linking to:
- `users`
- `items`

#### 1.1.7 Preview of the events table

We retrieve the first 5 rows of the events table and add row_count to estimate total volume.

Important: date and time are stored as character varying (text), which is not optimal, this must be considered in time-based analysis

```sql
SELECT 
    *,
    COUNT(*) OVER () AS row_count
FROM fantasy.events
LIMIT 5;
```

#### Query result

| transaction_id | id         | date       | time     | item_code | amount | seller_id | row_count |
|----------------|------------|------------|----------|-----------|--------|-----------|-----------|
| 2129235853     | 37-5938126 | 2021-01-03 | 16:31:49 | 6010      | 21.41  | 220381    | 1307678   |
| 2129237617     | 37-5938126 | 2021-01-03 | 16:49:00 | 6010      | 64.98  | 54680     | 1307678   |
| 2129239381     | 37-5938126 | 2021-01-03 | 21:05:29 | 6010      | 50.68  | 888909    | 1307678   |
| 2129241145     | 37-5938126 | 2021-01-03 | 22:03:02 | 6010      | 46.49  | 888902    | 1307678   |
| 2129242909     | 37-5938126 | 2021-01-03 | 22:04:26 | 6010      | 18.72  | 888905    | 1307678   |

The dataset contains over 1.3 million purchase records.

Also note: seller_id format differs from id, this likely indicates a separate seller registration system.

#### 1.1.8 Missing values in the events table

We check for missing values in key analytical fields: `date`, `time`, `amount`, `seller_id`.

```sql
SELECT COUNT(*) AS rows_with_nulls
FROM fantasy.events
WHERE date IS NULL
   OR time IS NULL
   OR amount IS NULL
   OR seller_id IS NULL;
```

**Query result**

| rows_with_nulls |
|-----------------|
| 508186          |

There are 508,186 rows with at least one missing value (out of 1,307,678).


#### 1.1.9 Detailed analysis of missing values

We further analyze missing values by column: `date_count`, `time_count`, `amount_count`, `seller_id_count`.

```sql
SELECT 
    COUNT(date) AS data_count,
    COUNT(time) AS time_count,
    COUNT(amount) AS amount_count,
    COUNT(seller_id) AS seller_id_count
FROM fantasy.events;
```

**Query result**

| data_count | time_count | amount_count | seller_id_count |
|------------|------------|--------------|----------------|
| 1307678    | 1307678    | 799492       | 1307678        |

All 508,186 missing values are in the seller_id field.

Other fields (`date`, `time`, `amount`) are complete.

Interpretation: Missing seller_id likely indicates purchases made via the in-game store, not from other players.

#### 1.2.1  Payers share

#### Overall share of paying users

```sql
WITH pc AS
(SELECT id,
        COUNT(*) AS payer_count
FROM fantasy.users
WHERE payer=1
GROUP BY id)
SELECT COUNT(u.id) AS user_count,
       SUM(payer_count) AS payers_count,
       ROUND(SUM(payer_count)::NUMERIC/COUNT(u.id), 6) AS ratio
FROM fantasy.users u
LEFT JOIN pc ON pc.id=u.id
```

**Query result**

| user_count | payers_count | ratio   |
|------------|--------------|---------|
| 22214      | 3929         | 0.17687 |

####  Share of paying users by character race

```sql
WITH pc AS
(SELECT id,
        COUNT(*) AS payer_count
FROM fantasy.users
WHERE payer=1
GROUP BY id)
SELECT race,
       SUM(payer_count) AS payers_count,
       COUNT(u.id) AS user_count,
       ROUND(SUM(payer_count)::NUMERIC/COUNT(u.id), 6) AS ratio
FROM fantasy.users u
LEFT JOIN pc ON pc.id=u.id
LEFT JOIN fantasy.race AS r ON r.race_id=u.race_id
GROUP BY race
ORDER BY ratio DESC
```

**Query result**

| race     | payers_count | user_count | ratio     |
|----------|--------------|------------|-----------|
| Demon    | 238          | 1229       | 0.193653  |
| Hobbit   | 659          | 3648       | 0.180647  |
| Human    | 1114         | 6328       | 0.176043  |
| Northman | 626          | 3562       | 0.175744  |
| Orc      | 636          | 3619       | 0.175739  |
| Angel    | 229          | 1327       | 0.172570  |
| Elf      | 427          | 2501       | 0.170732  |

#### 1.3 In-game purchases analysis

#### 1.3.1 Statistical metrics for `amount`

Key statistics (min, max, avg, median, std)

```sql
SELECT COUNT(amount) AS purchase_count,
       SUM(amount) AS total_amount,
       MAX(amount) AS max_purchase,
       MIN(amount) AS min_purchase,
       ROUND(AVG(amount::NUMERIC),2) AS avg_purchase,
       PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY amount) AS median_purchase,
       ROUND(STDDEV(amount::NUMERIC),2) AS stad_dev
FROM fantasy.events
```

#### Query result

| purchase_count | total_amount | max_purchase | min_purchase | avg_purchase | median_purchase | std_dev  |
|----------------|-------------|--------------|--------------|--------------|----------------|----------|
| 1307678        | 686615040   | 486615.1     | 0.0          | 525.69       | 74.86          | 2517.35  |

#### 1.3.2 Calculation of minimum purchase excluding anomalies

```sql
SELECT game_items,
       MIN(amount) AS min_purchase
FROM fantasy.events
LEFT JOIN fantasy.items USING(item_code)
WHERE amount>0
GROUP BY game_items
ORDER BY min_purchase
```

#### Query result

<div style="text-align: left;">

| game_items               | min_purchase |
|--------------------------|--------------|
| Book of Legends          | 0.01         |
| Quiver of Endless Arrows | 0.03         |
| Treasure Map             | 0.04         |
| Gauntlets of Might       | 0.05         |
| Ring of Invisibility     | 0.05         |
| Runes of Power           | 0.07         |
| Necklace of Wisdom       | 0.07         |
| Magic Animal             | 0.07         |
| Griffin Feather          | 0.07         |
| Water of Life            | 0.07         |
| Spectral Shield          | 2552.58      |

</div>

#### 1.3.3 Anomalies

Identification of anomalous zero-value transactions

```sql
WITH cte AS(
SELECT COUNT(amount) AS null_purchases
FROM fantasy.events
WHERE amount=0),
cte2 AS(
SELECT COUNT(amount) AS total_purchases
FROM fantasy.events)
SELECT null_purchases,
       total_purchases,
       ROUND((null_purchases::NUMERIC/total_purchases),6) AS ratio
FROM cte 
CROSS JOIN cte2
```

**Query result**

| null_purchases | total_purchases | ratio    |
|----------------|----------------|----------|
| 907            | 1307678        | 0.000694 |

#### 1.3.4 Items with zero purchase value

```sql
SELECT game_items,
       amount,
       COUNT(amount)
FROM fantasy.events e
RIGHT JOIN fantasy.items i USING(item_code)
WHERE amount=0
GROUP BY game_items, amount
```

**Query result**

| game_items          | amount | count |
|--------------------|--------|-------|
| Book of Legends     | 0.0    | 907   |

#### 1.3.5 Payer vs non-payer behavior

#### Comparison of activity between paying and non-paying users

```sql
SELECT  CASE WHEN payer = 0 THEN 'non_payer_user' ELSE 'payer_user' END AS payer_type,
  	    COUNT(DISTINCT e.id) AS users,
  	    ROUND(COUNT(*)::NUMERIC/ COUNT(DISTINCT e.id),2) AS avg_purchase_count,
  	    ROUND(SUM(amount)::NUMERIC/ count(DISTINCT e.id), 2) AS avg_purchase_sum
FROM fantasy.events e 
LEFT JOIN fantasy.users AS u ON u.id=e.id
WHERE amount<>0
GROUP BY payer
```

#### Query result

| payer_type      | users | avg_purchase_count | avg_purchase_sum |
|-----------------|-------|------------------|----------------|
| non_payer_user  | 11348 | 97.56            | 48579.66       |
| payer_user      | 2444  | 81.68            | 55469.72       |

#### Comparison by character race

```sql
SELECT  race,
        CASE WHEN payer = 0 THEN 'non_payer_user' ELSE 'payer_user' END AS payer_type,
  	COUNT(DISTINCT e.id) AS users,
  	ROUND(COUNT(*)::NUMERIC/COUNT(DISTINCT e.id),2) AS avg_purchase_count,
  	ROUND(SUM(amount)::NUMERIC/count(DISTINCT e.id), 2) AS avg_purchase_sum
FROM fantasy.events e 
LEFT JOIN fantasy.users AS u ON u.id=e.id
LEFT JOIN fantasy.race USING(race_id)
WHERE amount<>0
GROUP BY race, payer
```

#### Query result

| race    | payer_type     | users | avg_purchase_count | avg_purchase_sum |
|---------|---------------|-------|------------------|----------------|
| Angel   | non_payer_user| 683   | 100.12           | 49527.38       |
| Angel   | payer_user    | 137   | 140.14           | 44359.93       |
| Demon   | non_payer_user| 590   | 77.20            | 43718.47       |
| Demon   | payer_user    | 147   | 80.57            | 31069.59       |
| Elf     | non_payer_user| 1292  | 81.16            | 54315.17       |
| Elf     | payer_user    | 251   | 66.59            | 50908.76       |
| Hobbit  | non_payer_user| 1865  | 84.35            | 48690.29       |
| Hobbit  | payer_user    | 401   | 94.42            | 42650.37       |
| Human   | non_payer_user| 3215  | 127.54           | 49304.20       |
| Human   | payer_user    | 706   | 93.43            | 47257.08       |
| Northman| non_payer_user| 1823  | 88.43            | 50672.85       |
| Northman| payer_user    | 406   | 53.68            | 115728.33      |
| Orc     | non_payer_user| 1880  | 84.99            | 42745.32       |
| Orc     | payer_user    | 396   | 66.29            | 37084.60       |

#### 1.4 Popular epic items
#### 1.4.1 Most frequently purchased items

```sql
WITH cte AS (
SELECT game_items,
       COUNT(amount) AS sales_count,
       COUNT(DISTINCT e.id) AS buyer_users
FROM fantasy.items i
LEFT JOIN fantasy.events e USING(item_code)
WHERE amount>0
GROUP BY game_items)
SELECT game_items,
       sales_count,
       ROUND(sales_count::NUMERIC/SUM(sales_count) OVER(), 6) AS sales_ratio,
       ROUND(buyer_users::NUMERIC/(SELECT COUNT(DISTINCT id) FROM fantasy.events WHERE amount>0), 6) AS users_ratio
FROM cte
ORDER BY sales_ratio DESC
```

**Query result**

| game_items          | sales_count | sales_ratio | users_ratio |
|--------------------|------------|------------|------------|
| Book of Legends     | 1004516    | 0.768701   | 0.884136   |
| Bag of Holding      | 271875     | 0.208051   | 0.867749   |
| Necklace of Wisdom  | 13828      | 0.010582   | 0.117967   |
| Gems of Insight     | 3833       | 0.002933   | 0.067140   |
| Treasure Map        | 3183       | 0.002436   | 0.059382   |
| ...                 | ...        | ...        | ...        |
| Magic Box           | 11         | 0.000008   | 0.000725   |

#### 1.4.2 Items that were never purchased

```sql
WITH cte AS (
SELECT game_items,
       COUNT(amount) AS sales_count
FROM fantasy.items i
LEFT JOIN fantasy.events e USING(item_code)
GROUP BY game_items)
SELECT game_items
FROM cte
WHERE sales_count<1
```

#### Query result

| game_items              | game_items             | game_items             | game_items           |
|-------------------------|-----------------------|-----------------------|---------------------|
| Bard's Lute             | Cyclops Eye           | Lich's Phylactery     | Seer's Crystal Ball |
| Mirror of Reflection    | Torch of Eternal Flame| Alchemist's Cookbook  | Shaman's Drum       |
| Amulet of Defense       | Gargoyle Wing         | Phoenix Ashes         | Angel's Feather     |
| Rings of Teleportation  | Vials of Poison       | Goblin Ear            | Survival Kit        |
| Potion of Healing       | Mystic Chest          | Battle Axe            | Troll Blood         |
| Witch's Broom           | Giant's Toenail       | Harpy Feather         | Ring of Fire Resistance |
| Dwarf Beard             | Basilisk Eye          | Paralysis Dust        | Golem's Heart       |
| Star Map                | Sage's Ring           | Enchanted Sword       | Scroll of Time Stop |
| Ranger's Bow            | Staff of Fire         | Books of Lore         | Dagger of Stealth   |
| Magic Box               | ...                   | ...                   | ...                 |

### 2. Ad hoc analysis
#### 2.1  Player activity and purchasing behavior by race

This analysis explores how player activity in purchasing epic items varies across different character races.

The underlying hypothesis is that gameplay difficulty may differ by race, which could lead to variations in purchasing behavior. If certain races require more frequent purchases to progress, this may indicate an imbalance in game design. 

The objective is to identify such differences and assess whether gameplay is consistent across all races.

The analysis focuses on the following metrics for each race:

- total number of registered players
- number of players making in-game purchases and their share of total players
- share of paying users among players who make purchases
- average number of purchases per purchasing player
- average purchase value per purchasing player
- average total spending per purchasing player

This approach provides insight into both player engagement (purchase frequency) and monetization (spending levels), helping evaluate whether certain races require more effort or cost to progress.

```sql
WITH race_c AS(
SELECT r.race,
       COUNT(u.id) AS race_count
FROM fantasy.race r
LEFT JOIN fantasy.users u ON r.race_id=u.race_id
GROUP BY r.race),
race_payers AS(
SELECT r.race,
       COUNT(DISTINCT e.id) AS buyers_users,
       ROUND(COUNT(DISTINCT e.id)::NUMERIC/COUNT(DISTINCT u.id), 6) 
       AS buyers_from_all_ratio,
       ROUND(COUNT(DISTINCT e.id) FILTER (WHERE u.payer=1)/COUNT(DISTINCT e.id)::NUMERIC, 6)
       AS payers_from_buyers_ratio
FROM fantasy.race r
LEFT JOIN fantasy.users u ON r.race_id=u.race_id
LEFT JOIN fantasy.events e ON e.id=u.id
GROUP BY r.race),
counts AS(
SELECT r.race,
       ROUND(COUNT(e.transaction_id)::NUMERIC/COUNT(DISTINCT e.id),2) AS avg_purchases_count_per_user,
       ROUND(SUM(e.amount)::NUMERIC /COUNT(e.id),2) AS avg_amount_per_user,
       ROUND(SUM(e.amount)::NUMERIC/COUNT(DISTINCT e.id),2)  AS avg_total_amount_per_user
FROM fantasy.race r
LEFT JOIN fantasy.users u ON r.race_id=u.race_id
LEFT JOIN fantasy.events e ON e.id=u.id
WHERE amount<>0
GROUP BY r.race)
SELECT race,
       race_count,
       buyers_users,
       buyers_from_all_ratio,
       payers_from_buyers_ratio,
       avg_purchases_count_per_user,
       avg_amount_per_user,
       avg_total_amount_per_user
FROM race_c AS rc
LEFT JOIN race_payers AS rp USING(race)
LEFT JOIN counts AS c USING(race)
ORDER BY payers_from_buyers_ratio DESC
```

#### Query result

| race     | race_count | buyers_users | buyers_from_all_ratio | payers_from_buyers_ratio | avg_purchases_count_per_user | avg_amount_per_user | avg_total_amount_per_user |
|----------|------------|--------------|---------------------|-------------------------|-----------------------------|-------------------|--------------------------|
| Demon    | 1229       | 737          | 0.599675            | 0.199457                | 77.87                       | 529.02            | 41194.71                 |
| Northman | 3562       | 2229         | 0.625772            | 0.182144                | 82.10                       | 761.52            | 62522.21                 |
| Human    | 6328       | 3921         | 0.619627            | 0.180056                | 121.40                      | 403.05            | 48931.65                 |
| Hobbit   | 3648       | 2267         | 0.621436            | 0.176886                | 86.13                       | 552.91            | 47621.80                 |
| Orc      | 3619       | 2276         | 0.628903            | 0.173989                | 81.74                       | 510.92            | 41761.60                 |
| Angel    | 1327       | 820          | 0.617935            | 0.167073                | 106.80                      | 455.64            | 48664.88                 |
| Elf      | 2501       | 1543         | 0.616953            | 0.162670                | 78.79                       | 682.32            | 53760.40                 |

### 3. Conclusions & analytical insights

**3.1 Payer share**

The overall share of paying users is **17.68%** of the total user base (**3,929** out of **22,214** users).
The highest payer share is observed among:
- Demon — **19.36%**
- Hobbit — **18.04%**

At the same time, Demon is the least selected race among players.

The lowest payer share is observed for:
- Elf — **17.07%**
- Angel — **17.26%**

This indicates that player monetization behavior varies depending on character race.

**3.2 In-game purchases analysis**

A total of **1,307,678** purchases were made, generating **686,615,040 RUB** in revenue.

- Minimum purchase value: **0.01**
- Maximum purchase value: **486,615.1**

Both extremes are associated with the same item: Book of Legends

The average purchase value **(525.69)** is significantly higher than the median **(74.86)**, indicating a strong right-skewed distribution:

- most purchases are low-value
- a small number of high-value transactions significantly increase the average

The standard deviation **(2517.35)** is substantially higher than the mean, confirming high variability and the presence of expensive outliers.

There are **907** zero-value purchases, representing **0.07%** of all transactions.
All zero-value purchases are associated with Book of Legends and should be considered anomalies and excluded from further analysis.

**3.3 Purchasing behavior**

- A total of **13,793** users made purchases.
- Non-paying users: **11,349**
- Paying users: **2,444**

Non-paying users make significantly more purchases:
- **97.56** purchases per user (non-payers)
- **81.68** purchases per user (payers)

However, paying users generate higher revenue:

- **55,467.68** average total spend per user (payers)
- **48,588.47** (non-payers)

Additionally, paying users of the Northman race spend more than twice as much per purchase compared to non-paying users and other races.

**3.4 Item popularity**

Two items dominate purchases:

- Book of Legends — **76.87%** of all purchases
- Bag of Holding — **20.81%**

These two items account for the vast majority of in-game transactions.
There are also **38** items that were never purchased, indicating low or no player interest.

#### 3.5 Player activity and purchasing behavior by race

- The most active players in terms of purchases are **Orc** and **Northman**
- The lowest share of paying users among purchasers is observed for **Elf** and **Angel**
- The Demon race shows:**lower purchase frequency, but the highest share of paying users**
- Players of Northman and Elf races demonstrate the highest **average purchase value and total spending per user**

### Final conclusions & recommendations

1. Demon and Hobbit have the highest payer share, indicating strong monetization potential. **Recommendation:** target these segments with personalized offers and premium currency bundles.


2. Northman players spend more per purchase and have strong monetization potential.**Recommendation:** introduce mechanics that encourage small but frequent payments (e.g., premium quests or progression boosts).


3. Elf players show low payer share but high spending among purchasers. **Hypothesis:** players may rely more on in-game advantages rather than paid currency. **Recommendation:** review game balance and incentives for this segment.


4. There are 38 unused items that generate no revenue. **Recommendation:** consider removing or redesigning them.


5. Book of Legends dominates purchases and includes zero-value transactions. **Recommendation:** investigate anomalies and consider introducing a premium version of this item to increase monetization.